In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Mount Google Drive**

Connects Google Colab to Google Drive to access files stored there.

Allows reading audio files from Drive and saving processed data/models back.



In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

# Paths
audio_dir = '/content/drive/MyDrive/stress_data_labeled/audio'
data = []

# Extract MFCC features from each file
for label in ['relaxed', 'stressed']:
    label_dir = os.path.join(audio_dir, label)
    for filename in os.listdir(label_dir):
        if filename.endswith('.wav'):
            filepath = os.path.join(label_dir, filename)
            y, sr = librosa.load(filepath, sr=22050)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            mfcc_mean = np.mean(mfcc, axis=1)  # 13 MFCC features
            data.append([*mfcc_mean, label])

# Convert to DataFrame
columns = [f"mfcc_{i+1}" for i in range(13)] + ["label"]
df = pd.DataFrame(data, columns=columns)

# Save to CSV
df.to_csv('/content/drive/MyDrive/stress_audio_features.csv', index=False)
print("✅ MFCC features saved.")


✅ MFCC features saved.


** Extract MFCC Features from Audio Files**

 Loads Audio Files: Reads .wav files from relaxed and stressed folders.

Extracts MFCC Features: Computes 13 Mel-Frequency Cepstral Coefficients (MFCCs) per audio file (commonly used in speech/sound analysis).

Averages MFCCs: Takes the mean of each coefficient across time to get a fixed-length feature vector.

Stores Data: Appends features along with their label (relaxed or stressed) in a list.

**Save as CSV file**

Converts the list of features into a Pandas DataFrame.

Saves it as a CSV file (stress_audio_features.csv) for later use (avoiding reprocessing).

In [ ]:
import pandas as pd

df_audio = pd.read_csv('/content/drive/MyDrive/stress_audio_features.csv')
print(df_audio.head())


      mfcc_1     mfcc_2     mfcc_3     mfcc_4    mfcc_5    mfcc_6     mfcc_7  \
0 -458.98343  44.221943 -23.751170   9.087063 -5.701322 -8.484038 -21.735100   
1 -473.86496  49.355560 -22.830324   8.363604 -5.262116 -5.796022 -19.709543   
2 -537.75397  58.140040 -22.289553  12.773673  0.548504 -5.221027 -16.057827   
3 -566.80457  54.642906 -12.577813  10.267268 -2.378791 -5.390403  -8.305521   
4 -579.50336  51.303246 -13.493130  10.219903 -1.105671 -8.917431  -8.942595   

      mfcc_8     mfcc_9    mfcc_10   mfcc_11    mfcc_12   mfcc_13    label  
0  -6.596186 -19.839092  -8.206760 -1.430951 -10.099604 -2.321979  relaxed  
1  -8.201800 -20.104876 -11.078058  0.140050 -12.031988 -3.396361  relaxed  
2  -5.860365 -21.034155  -8.284555  1.113632 -12.508387 -3.215678  relaxed  
3 -10.356411 -20.050936  -6.694321 -1.810556  -7.454474 -4.242743  relaxed  
4 -10.207701 -17.658115  -9.292863 -4.208510  -7.848867 -4.908560  relaxed  


**Load & Preview Data**

Reads the saved CSV file back into a DataFrame.

Displays the first few rows to verify the data structure (13 MFCCs + label).

In [ ]:
X = df_audio.drop(columns=['label'])  # only drop 'label'
y = df_audio['label'].map({'relaxed': 0, 'stressed': 1})


What it does:

Takes the DataFrame df_audio (which contains MFCC features + label) and removes the 'label' column.

The remaining columns (mfcc_1 to mfcc_13) are stored in X, which becomes the feature matrix (input for the ML model).


Takes the 'label' column from df_audio and converts it into binary numerical values:

'relaxed' → 0

'stressed' → 1

The result (y) is the target vector for the model.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Drop only the label column
X = df_audio.drop(columns=['label'])
y = df_audio['label'].map({'relaxed': 0, 'stressed': 1})  # binary encoding

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Relaxed', 'Stressed']))


              precision    recall  f1-score   support

     Relaxed       0.89      0.90      0.89       201
    Stressed       0.93      0.92      0.93       290

    accuracy                           0.91       491
   macro avg       0.91      0.91      0.91       491
weighted avg       0.91      0.91      0.91       491



**Splitting the Dataset**

Splits the dataset (X = features, y = labels) into training (80%) and testing (20%) sets.

random_state=42 ensures reproducibility (same split every time the code runs).

**Training the Random Forest Model **

Initializes a Random Forest classifier with:

n_estimators=100: 100 decision trees in the ensemble.

random_state=42: Ensures consistent results.

Trains the model on X_train (MFCC features) and y_train (labels).

**Making Predictions**

Uses the trained model (clf) to predict labels for the test set (X_test).

Outputs y_pred (predicted labels: 0 = relaxed, 1 = stressed).

**Evaluating the Model**

Generates a classification report with key metrics:

Precision: % of correct predictions for each class.

Recall: % of actual class instances correctly predicted.

F1-score: Harmonic mean of precision and recall.

Support: Number of samples in each class.

In [ ]:
import os
import joblib

# Create directory if it doesn't exist
os.makedirs('/content/drive/MyDrive/model', exist_ok=True)

# Save the model
joblib.dump(clf, '/content/drive/MyDrive/model/stress_model_audio.pkl')
print("✅ Model saved.")


✅ Model saved.


** Save & Download the Model**

Saves Model: Stores the trained Random Forest as a .pkl file in Drive.

Downloads Model: Allows you to download the model file locally for deployment elsewhere.



In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/model/stress_model_audio.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Imports the files module from google.colab:

This module provides tools for file management in Google Colab, including uploading/downloading files.

Downloads the trained model (stress_model_audio.pkl) from Google Drive to your local machine:

The file is fetched from the path /content/drive/MyDrive/model/stress_model_audio.pkl (where it was saved earlier).

Triggers a browser download dialog to save the .pkl file locally.

